# 02 — Data Cleaning and Preparation

## Objective

Clean and standardize the validated CFPB dataset, assess data quality,
prepare complaint narratives, and create a reproducible balanced
analytical dataset for downstream EDA, NLP, and RAG.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_PATH = Path("/Users/ishika/Documents/AIProject/data/interim/raw_combined_data.csv")

df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Dataset shape: {df.shape}")
display(df.head())
df.info()

Dataset shape: (566694, 17)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,source_file
0,2025-01-02 14:26:35+00:00,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,JPMORGAN CHASE & CO.,TN,37066,NaN,Phone,2025-01-02T14:41:19.000Z,Closed with explanation,Yes,11354262,CheckingsOrSavings.csv
1,2025-01-01 02:34:28+00:00,Checking or savings account,Checking account,Managing an account,Funds not handled or disbursed as instructed,Are deposits made to Bank of America actually ...,Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",TN,37217,NaN,Web,2025-01-01T02:50:13.000Z,Closed with explanation,Yes,11345823,CheckingsOrSavings.csv
2,2025-01-01 00:54:23+00:00,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,JPMORGAN CHASE & CO.,SC,29306,Servicemember,Web,2025-01-01T01:17:26.000Z,Closed with explanation,Yes,11334911,CheckingsOrSavings.csv
3,2025-01-01 02:20:18+00:00,Checking or savings account,Savings account,Opening an account,Account opened without my consent or knowledge,NaN,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,NY,10037,NaN,Web,2025-01-01T02:53:35.000Z,Closed with explanation,Yes,11346230,CheckingsOrSavings.csv
4,2025-01-01 03:00:01+00:00,Checking or savings account,Checking account,Problem caused by your funds being low,Overdrafts and overdraft fees,NaN,Company believes it acted appropriately as aut...,NAVY FEDERAL CREDIT UNION,MO,65201,Servicemember,Web,2025-01-01T03:09:05.000Z,Closed with explanation,Yes,11334580,CheckingsOrSavings.csv


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566694 entries, 0 to 566693
Data columns (total 17 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Date received                 566694 non-null  object
 1   Product                       566694 non-null  object
 2   Sub-product                   566694 non-null  object
 3   Issue                         566694 non-null  object
 4   Sub-issue                     481917 non-null  object
 5   Consumer complaint narrative  272466 non-null  object
 6   Company public response       193788 non-null  object
 7   Company                       566694 non-null  object
 8   State                         563712 non-null  object
 9   ZIP code                      566694 non-null  object
 10  Tags                          55463 non-null   object
 11  Submitted via                 566694 non-null  object
 12  Date sent to company          566694 non-null  object
 13 

In [3]:
# Basic dataset inspection

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("data_type"))

Dataset shape: (566694, 17)

Column names:
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID', 'source_file']

Data types:


,data_type
Date received,object
Product,object
Sub-product,object
Issue,object
Sub-issue,object
Consumer complaint narrative,object
Company public response,object
Company,object
State,object
ZIP code,object


In [4]:
# Missing-value analysis

missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percentage", ascending=False)

display(missing_report)

,missing_count,missing_percentage
Tags,511231,90.21
Company public response,372906,65.80
Consumer complaint narrative,294228,51.92
Sub-issue,84777,14.96
State,2982,0.53
Complaint ID,0,0.00
Timely response?,0,0.00
Company response to consumer,0,0.00
Date sent to company,0,0.00
Submitted via,0,0.00


In [5]:
# Standardize column names

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace("?", "", regex=False)
      .str.replace("-", "_", regex=False)
      .str.replace(" ", "_", regex=False)
)

print(df.columns.tolist())

['date_received', 'product', 'sub_product', 'issue', 'sub_issue', 'consumer_complaint_narrative', 'company_public_response', 'company', 'state', 'zip_code', 'tags', 'submitted_via', 'date_sent_to_company', 'company_response_to_consumer', 'timely_response', 'complaint_id', 'source_file']


In [6]:
# Inspect current data types

data_types = df.dtypes.to_frame(name="data_type")
display(data_types)

,data_type
date_received,object
product,object
sub_product,object
issue,object
sub_issue,object
consumer_complaint_narrative,object
company_public_response,object
company,object
state,object
zip_code,object


In [7]:
# Convert date columns

date_columns = ["date_received", "date_sent_to_company"]

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

# Check whether any dates failed conversion
date_validation = pd.DataFrame({
    "missing_or_invalid": df[date_columns].isna().sum(),
    "minimum_date": df[date_columns].min(),
    "maximum_date": df[date_columns].max()
})

display(date_validation)

,missing_or_invalid,minimum_date,maximum_date
date_received,0,2025-01-01 00:00:29+00:00,2025-12-31 23:58:31+00:00
date_sent_to_company,0,2025-01-01 00:03:25+00:00,2026-08-03 18:42:48+00:00


In [8]:
invalid_date_order = df[
    df["date_sent_to_company"] < df["date_received"]
]

print("Rows with invalid date order:", len(invalid_date_order))
display(invalid_date_order.head())

Rows with invalid date order: 0


,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id,source_file


In [9]:
# Categorical text columns to standardize

categorical_columns = [
    "product",
    "sub_product",
    "issue",
    "sub_issue",
    "company_public_response",
    "company",
    "state",
    "tags",
    "submitted_via",
    "company_response_to_consumer",
    "timely_response",
    "source_file"
]

# Remove leading/trailing spaces and normalize internal whitespace
for column in categorical_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

In [10]:
columns_to_check = [
    "product",
    "submitted_via",
    "company_response_to_consumer",
    "timely_response"
]

for column in columns_to_check:
    print(f"\n{column}")
    print("-" * len(column))
    print(df[column].value_counts(dropna=False))


product
-------
product
Debt collection                                       283205
Credit card                                            89979
Money transfer, virtual currency, or money service     84619
Checking or savings account                            84194
Mortgage                                               24697
Name: count, dtype: Int64

submitted_via
-------------
submitted_via
Web            546114
Referral         9220
Phone            8750
Postal mail      2610
Name: count, dtype: Int64

company_response_to_consumer
----------------------------
company_response_to_consumer
Closed with explanation            443349
Closed with non-monetary relief     97002
Closed with monetary relief         23196
Untimely response                    3120
In progress                            27
Name: count, dtype: Int64

timely_response
---------------
timely_response
Yes    556143
No      10551
Name: count, dtype: Int64


In [11]:
empty_string_report = pd.Series({
    column: df[column].eq("").sum()
    for column in categorical_columns
}).sort_values(ascending=False)

display(empty_string_report.to_frame("empty_string_count"))

,empty_string_count
product,0
sub_product,0
issue,0
sub_issue,0
company_public_response,0
company,0
state,0
tags,0
submitted_via,0
company_response_to_consumer,0


In [12]:
df[categorical_columns] = df[categorical_columns].replace("", pd.NA)

In [13]:
print("Dataset shape:", df.shape)
print("\nProduct counts:")
display(df["product"].value_counts())

print("\nMissing Complaint IDs:", df["complaint_id"].isna().sum())
print("Duplicate Complaint IDs:", df["complaint_id"].duplicated().sum())

Dataset shape: (566694, 17)

Product counts:


product
Debt collection                                       283205
Credit card                                            89979
Money transfer, virtual currency, or money service     84619
Checking or savings account                            84194
Mortgage                                               24697
Name: count, dtype: Int64


Missing Complaint IDs: 0
Duplicate Complaint IDs: 0


In [14]:
# Create narrative-availability indicators

narrative_column = "consumer_complaint_narrative"

df["has_narrative"] = (
    df[narrative_column]
    .notna()
    & df[narrative_column].str.strip().ne("")
)

print("Total complaints:", len(df))
print("Complaints with narratives:", df["has_narrative"].sum())
print("Complaints without narratives:", (~df["has_narrative"]).sum())
print(
    "Narrative availability:",
    f"{df['has_narrative'].mean() * 100:.2f}%"
)

Total complaints: 566694
Complaints with narratives: 272466
Complaints without narratives: 294228
Narrative availability: 48.08%


In [15]:
narrative_by_product = (
    df.groupby("product", observed=True)
      .agg(
          total_complaints=("complaint_id", "count"),
          complaints_with_narrative=("has_narrative", "sum")
      )
)

narrative_by_product["complaints_without_narrative"] = (
    narrative_by_product["total_complaints"]
    - narrative_by_product["complaints_with_narrative"]
)

narrative_by_product["narrative_percentage"] = (
    narrative_by_product["complaints_with_narrative"]
    / narrative_by_product["total_complaints"]
    * 100
).round(2)

narrative_by_product = narrative_by_product.sort_values(
    "total_complaints",
    ascending=False
)

display(narrative_by_product)

,total_complaints,complaints_with_narrative,complaints_without_narrative,narrative_percentage
product,,,,
Debt collection,283205,100610,182595,35.53
Credit card,89979,41517,48462,46.14
"Money transfer, virtual currency, or money service",84619,64862,19757,76.65
Checking or savings account,84194,51448,32746,61.11
Mortgage,24697,14029,10668,56.80


### Narrative availability

Of the 566,694 complaints, 272,466 (48.08%) contain a consumer
complaint narrative. Narrative availability varies considerably across
products, ranging from 35.53% for debt collection to 76.65% for money
transfer and related services.

Complaints without narratives will not be removed from the structured
analytical dataset because they remain useful for SQL analysis and
business KPIs. Requiring a narrative would also introduce product-level
selection bias. Narrative-dependent analysis will instead use an
eligible subset of complaints containing usable narrative text.

In [16]:
# Narrative-length features

df["narrative_character_count"] = (
    df[narrative_column]
    .where(df["has_narrative"])
    .str.len()
)

df["narrative_word_count"] = (
    df[narrative_column]
    .where(df["has_narrative"])
    .str.split()
    .str.len()
)

In [17]:
narrative_length_summary = (
    df.loc[df["has_narrative"],
           ["narrative_character_count", "narrative_word_count"]]
      .describe(
          percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
      )
      .round(2)
)

display(narrative_length_summary)

,narrative_character_count,narrative_word_count
count,272466.00,272466.00
mean,1029.18,174.50
std,1149.04,194.66
min,39.00,4.00
1%,71.00,13.00
5%,150.00,27.00
25%,403.00,72.00
50%,678.00,107.00
75%,1265.00,218.00
95%,2876.75,489.00


In [18]:
narrative_length_by_product = (
    df.loc[df["has_narrative"]]
      .groupby("product", observed=True)["narrative_word_count"]
      .agg(
          narrative_count="count",
          minimum="min",
          median="median",
          mean="mean",
          maximum="max"
      )
      .round(2)
      .sort_values("narrative_count", ascending=False)
)

display(narrative_length_by_product)

,narrative_count,minimum,median,mean,maximum
product,,,,,
Debt collection,100610,6.0,102.0,160.74,5220.0
"Money transfer, virtual currency, or money service",64862,4.0,102.0,131.35,5699.0
Checking or savings account,51448,7.0,128.0,189.79,6218.0
Credit card,41517,5.0,169.0,214.50,6469.0
Mortgage,14029,8.0,218.0,298.19,5765.0


In [19]:
short_narratives = df.loc[
    df["has_narrative"] & (df["narrative_word_count"] < 10),
    ["complaint_id", "product", "narrative_word_count",
     "consumer_complaint_narrative"]
].sort_values("narrative_word_count")

print("Narratives containing fewer than 10 words:", len(short_narratives))
display(short_narratives.head(20))

Narratives containing fewer than 10 words: 679


,complaint_id,product,narrative_word_count,consumer_complaint_narrative
225902,11972006,"Money transfer, virtual currency, or money ser...",4.0,XX/XX/XXXX Cash App paymentXX/XX/XXXXXX/XX/XXXX
129426,14268747,Credit card,5.0,"unauthorized reporting, unverifiable ledger en..."
173580,18372027,Credit card,6.0,Received letter requesting more identity infor...
200268,11646905,"Money transfer, virtual currency, or money ser...",6.0,missing/unauthorized funds. no cash app assist...
192738,11569094,"Money transfer, virtual currency, or money ser...",6.0,Unauthorized transactions from my checking acc...
192642,11667711,"Money transfer, virtual currency, or money ser...",6.0,Company completed fraudulent transactions with...
340208,12626186,Debt collection,6.0,credence resource management collection unknow...
490033,14501695,Debt collection,6.0,INACCURATE INFORMATION FALSE INFORMATION BEING...
127395,14412980,Credit card,6.0,American Express calling me aggressively every...
512385,15023660,Debt collection,6.0,b'Collection \nMidland credit management inc \...


### Short-narrative assessment

Manual inspection showed that several narratives containing fewer than
10 words still expressed identifiable consumer issues. Therefore, no
minimum word-count threshold was imposed. Narrative length will be
retained as a descriptive feature rather than used as an automatic
row-removal criterion.

In [20]:
# Inspect common narrative formatting artifacts

narratives = df.loc[
    df["has_narrative"],
    "consumer_complaint_narrative"
].astype("string")

artifact_report = pd.Series({
    "starts_with_bytes_marker": narratives.str.match(
        r"""^b['"]""", na=False
    ).sum(),
    "contains_literal_newline": narratives.str.contains(
        r"\\n", regex=True, na=False
    ).sum(),
    "contains_actual_newline": narratives.str.contains(
        "\n", regex=False, na=False
    ).sum(),
    "contains_html_entity": narratives.str.contains(
        r"&(?:gt|lt|amp|quot|#39);",
        regex=True,
        case=False,
        na=False
    ).sum(),
    "contains_repeated_whitespace": narratives.str.contains(
        r"\s{2,}",
        regex=True,
        na=False
    ).sum(),
    "contains_redaction_marker": narratives.str.contains(
        r"\bX{2,}\b",
        regex=True,
        case=False,
        na=False
    ).sum()
}, name="narrative_count")

display(artifact_report.to_frame())

,narrative_count
starts_with_bytes_marker,1018
contains_literal_newline,1067
contains_actual_newline,116112
contains_html_entity,0
contains_repeated_whitespace,136496
contains_redaction_marker,169991


In [21]:
byte_marker_examples = df.loc[
    df["has_narrative"]
    & df["consumer_complaint_narrative"]
        .astype("string")
        .str.match(r"""^b['"]""", na=False),
    ["complaint_id", "product", "consumer_complaint_narrative"]
]

print("Narratives beginning with a byte marker:", len(byte_marker_examples))
display(byte_marker_examples.head(10))

Narratives beginning with a byte marker: 1018


,complaint_id,product,consumer_complaint_narrative
30779,13028565,Checking or savings account,b'Subject: Formal Request for Refund of XXXX i...
30792,13025360,Checking or savings account,b'My name is XXXX XXXX please add this new is...
30919,13047306,Checking or savings account,"b""I have had cash app for years I can't tell t..."
30997,13031797,Checking or savings account,b'I responded to a text on my phone ostensibly...
31369,13100368,Checking or savings account,b'Solicited by written offer of a bonus of XXX...
31825,13158689,Checking or savings account,"b""CHIME XXXX XXXX XXXX XXXXXXXX XXXXXXXX CA XX..."
32488,13233556,Checking or savings account,b'On XXXX XXXX I had a Hotel charge at the XXX...
32490,13233137,Checking or savings account,b'I am filing a complaint against Bank Of Amer...
32726,13293288,Checking or savings account,b'Capital one closed my account sir so they sa...
32811,13040111,Checking or savings account,"b""On XXXX XXXX, Citibank unepectedly blocked m..."


In [22]:
# Identify which source files contain byte-string artifacts

byte_marker_mask = (
    df["has_narrative"]
    & df["consumer_complaint_narrative"]
        .astype("string")
        .str.match(r"""^b['"]""", na=False)
)

print("Total byte-marker narratives:", byte_marker_mask.sum())

byte_markers_by_source = (
    df.loc[byte_marker_mask]
      .groupby(["source_file", "product"], observed=True)
      .size()
      .reset_index(name="affected_narratives")
      .sort_values("affected_narratives", ascending=False)
)

display(byte_markers_by_source)

Total byte-marker narratives: 1018


,source_file,product,affected_narratives
1,CreditCard2025.csv,Credit card,241
0,CheckingsOrSavings.csv,Checking or savings account,228
7,debtcollectionseptember.csv,Debt collection,149
2,MoneyTransfer.csv,"Money transfer, virtual currency, or money ser...",127
5,debtcollectiondecember.csv,Debt collection,113
3,Mortgage.csv,Mortgage,87
6,debtcollectionmay.csv,Debt collection,38
4,debtcollection31may.csv,Debt collection,35


In [23]:
source_artifact_summary = (
    df.groupby("source_file", observed=True)
      .agg(
          total_rows=("complaint_id", "count"),
          narrative_rows=("has_narrative", "sum")
      )
)

source_artifact_summary["byte_marker_rows"] = (
    df.loc[byte_marker_mask]
      .groupby("source_file", observed=True)
      .size()
      .reindex(source_artifact_summary.index, fill_value=0)
)

source_artifact_summary["affected_narrative_percentage"] = (
    source_artifact_summary["byte_marker_rows"]
    / source_artifact_summary["narrative_rows"]
    * 100
).round(2)

display(
    source_artifact_summary.sort_values(
        "byte_marker_rows",
        ascending=False
    )
)

,total_rows,narrative_rows,byte_marker_rows,affected_narrative_percentage
source_file,,,,
CreditCard2025.csv,89979,41517,241,0.58
CheckingsOrSavings.csv,84194,51448,228,0.44
debtcollectionseptember.csv,99175,38928,149,0.38
MoneyTransfer.csv,84619,64862,127,0.20
debtcollectiondecember.csv,80536,20700,113,0.55
Mortgage.csv,24697,14029,87,0.62
debtcollectionmay.csv,21959,9294,38,0.41
debtcollection31may.csv,81535,31688,35,0.11


In [24]:
import ast

def can_decode_byte_literal(value):
    try:
        decoded = ast.literal_eval(value)
        return isinstance(decoded, bytes)
    except (ValueError, SyntaxError):
        return False

byte_literal_validation = (
    df.loc[byte_marker_mask, "consumer_complaint_narrative"]
      .astype(str)
      .apply(can_decode_byte_literal)
      .value_counts(dropna=False)
)

display(byte_literal_validation)

consumer_complaint_narrative
True     688
False    330
Name: count, dtype: int64

In [25]:
# Isolate byte-marker narratives that cannot be decoded directly

byte_marker_values = (
    df.loc[byte_marker_mask, "consumer_complaint_narrative"]
      .astype(str)
)

decodable_mask = byte_marker_values.apply(can_decode_byte_literal)

invalid_byte_literals = df.loc[
    byte_marker_values.index[~decodable_mask],
    [
        "complaint_id",
        "source_file",
        "product",
        "consumer_complaint_narrative"
    ]
].copy()

print("Invalid byte-literal cases:", len(invalid_byte_literals))

Invalid byte-literal cases: 330


In [26]:
for value in invalid_byte_literals[
    "consumer_complaint_narrative"
].head(15):
    print("START:", repr(value[:120]))
    print("END:  ", repr(value[-120:]))
    print()

START: "b'I have a checking account and a savingd account with Wells Fargo Bank.  I regularly make instant transfers between the"
END:   ' ACH payment (creating a negative balance), it cannot later return it as uncollected unless the merchant reverses itXXXX'

START: 'b"I am the victim of a fraudulent scam that resulted in a total loss of XXXX in USDC (crypto). On or around [insert date'
END:   ' XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX'

START: "b'Date occurred: XXXX XXXX 2025 XXXX  at approximately XXXX XXXX\\n\\nSubject: Formal Complaint Regarding Unprofessional C"
END:   'd to take to address this issue. Thank you for your attention to this matter. \\n\\nSincerely, \\nXXXX XXXX\\nXXXX XXXX\\XXXX'

START: "b'am reaching out regarding a payment of XXXX that was originally disputed through Chase Bank. Chase has informed me tha"
END:   ' and regulatory channels if necessary.XXXXThank you for your immediate a

In [28]:
def byte_wrapper_status(value):
    value = str(value)

    if value.startswith("b'"):
        opening_quote = "'"
    elif value.startswith('b"'):
        opening_quote = '"'
    else:
        return "no_byte_prefix"

    return (
        "matching_closing_quote"
        if value.endswith(opening_quote)
        else "missing_closing_quote"
    )

wrapper_status = (
    invalid_byte_literals["consumer_complaint_narrative"]
      .apply(byte_wrapper_status)
      .value_counts()
)

display(wrapper_status)

consumer_complaint_narrative
missing_closing_quote     329
matching_closing_quote      1
Name: count, dtype: int64

In [29]:
import ast
import html
import re
import pandas as pd

def clean_complaint_narrative(value):
    """
    Clean formatting artifacts while preserving the narrative's meaning
    and CFPB redaction markers.
    """
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    # Handle narratives serialized as Python byte strings
    if text.startswith(("b'", 'b"')):
        opening_quote = text[1]

        try:
            decoded = ast.literal_eval(text)

            if isinstance(decoded, bytes):
                text = decoded.decode("utf-8", errors="replace")
        except (ValueError, SyntaxError):
            # Malformed byte wrapper: remove b' or b"
            text = text[2:]

            # Remove closing quote only when present
            if text.endswith(opening_quote):
                text = text[:-1]

    # Convert HTML entities such as &gt; and &amp;
    text = html.unescape(text)

    # Convert escaped formatting characters into spaces
    text = (
        text.replace("\\n", " ")
            .replace("\\r", " ")
            .replace("\\t", " ")
    )

    # Normalize actual whitespace without changing wording
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else pd.NA

In [30]:
df["cleaned_narrative"] = (
    df["consumer_complaint_narrative"]
      .apply(clean_complaint_narrative)
      .astype("string")
)

In [31]:
cleaning_validation = pd.Series({
    "original_non_missing":
        df["consumer_complaint_narrative"].notna().sum(),

    "cleaned_non_missing":
        df["cleaned_narrative"].notna().sum(),

    "remaining_byte_markers":
        df["cleaned_narrative"]
          .str.match(r"""^b['"]""", na=False)
          .sum(),

    "remaining_literal_newlines":
        df["cleaned_narrative"]
          .str.contains(r"\\n", regex=True, na=False)
          .sum(),

    "remaining_html_entities":
        df["cleaned_narrative"]
          .str.contains(
              r"&(?:gt|lt|amp|quot|#39);",
              regex=True,
              case=False,
              na=False
          )
          .sum(),

    "empty_cleaned_narratives":
        df["cleaned_narrative"].eq("").sum()
})

display(cleaning_validation.to_frame("count"))

,count
original_non_missing,272466
cleaned_non_missing,272466
remaining_byte_markers,0
remaining_literal_newlines,0
remaining_html_entities,0
empty_cleaned_narratives,0


In [32]:
# Inspect before and after results
comparison = df.loc[
    byte_marker_mask,
    [
        "complaint_id",
        "consumer_complaint_narrative",
        "cleaned_narrative"
    ]
].head(10)

display(comparison)

,complaint_id,consumer_complaint_narrative,cleaned_narrative
30779,13028565,b'Subject: Formal Request for Refund of XXXX i...,Subject: Formal Request for Refund of XXXX in ...
30792,13025360,b'My name is XXXX XXXX please add this new is...,My name is XXXX XXXX please add this new issue...
30919,13047306,"b""I have had cash app for years I can't tell t...",I have had cash app for years I can't tell the...
30997,13031797,b'I responded to a text on my phone ostensibly...,I responded to a text on my phone ostensibly f...
31369,13100368,b'Solicited by written offer of a bonus of XXX...,Solicited by written offer of a bonus of XXXX ...
31825,13158689,"b""CHIME XXXX XXXX XXXX XXXXXXXX XXXXXXXX CA XX...",CHIME XXXX XXXX XXXX XXXXXXXX XXXXXXXX CA XXXX...
32488,13233556,b'On XXXX XXXX I had a Hotel charge at the XXX...,On XXXX XXXX I had a Hotel charge at the XXXXX...
32490,13233137,b'I am filing a complaint against Bank Of Amer...,I am filing a complaint against Bank Of Americ...
32726,13293288,b'Capital one closed my account sir so they sa...,Capital one closed my account sir so they said...
32811,13040111,"b""On XXXX XXXX, Citibank unepectedly blocked m...","On XXXX XXXX, Citibank unepectedly blocked my ..."


In [33]:
# Recalculate the cleaner evrsion
df["has_cleaned_narrative"] = df["cleaned_narrative"].notna()

df["narrative_character_count"] = (
    df["cleaned_narrative"].str.len().astype("Int64")
)

df["narrative_word_count"] = (
    df["cleaned_narrative"].str.split().str.len().astype("Int64")
)

### Narrative-format cleaning

Some complaint narratives contained byte-string serialization markers,
escaped newline characters, and HTML entities. A conservative cleaning
function removed these formatting artifacts and normalized whitespace
while preserving the original narrative column, substantive wording,
punctuation, and CFPB redaction markers.

All 272,466 available narratives were retained. No cleaned narrative
became missing or empty.

In [34]:
# Create a comparison-only version for detecting repeated narratives

narrative_comparison = (
    df["cleaned_narrative"]
      .str.lower()
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

narrative_duplicate_mask = (
    narrative_comparison.notna()
    & narrative_comparison.duplicated(keep=False)
)

print(
    "Rows belonging to repeated-narrative groups:",
    narrative_duplicate_mask.sum()
)

print(
    "Unique repeated narrative texts:",
    narrative_comparison[narrative_duplicate_mask].nunique()
)

Rows belonging to repeated-narrative groups: 62453
Unique repeated narrative texts: 5051


In [35]:
repeated_narrative_summary = (
    df.loc[narrative_duplicate_mask]
      .assign(narrative_key=narrative_comparison[narrative_duplicate_mask])
      .groupby("narrative_key", observed=True)
      .agg(
          occurrence_count=("complaint_id", "count"),
          unique_complaint_ids=("complaint_id", "nunique"),
          unique_companies=("company", "nunique"),
          unique_products=("product", "nunique"),
          example_product=("product", "first"),
          example_company=("company", "first"),
          example_narrative=("cleaned_narrative", "first")
      )
      .sort_values("occurrence_count", ascending=False)
      .reset_index(drop=True)
)

display(repeated_narrative_summary.head(20))

,occurrence_count,unique_complaint_ids,unique_companies,unique_products,example_product,example_company,example_narrative
0,9976,9976,2,2,Checking or savings account,"Block, Inc.",I am filing a complaint against Cash App ( Blo...
1,5591,5591,8,1,Debt collection,"EQUIFAX, INC.",There are collection accounts on my report tha...
2,4315,4315,3,2,Checking or savings account,"Early Warning Services, LLC",I am writing to formally express my concerns r...
3,3123,3123,7,1,Debt collection,"EQUIFAX, INC.",I understand the importance of removing any in...
4,3009,3009,3,1,Debt collection,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",I want to stress that I did not give written p...
5,2507,2507,1,2,Checking or savings account,NAVY FEDERAL CREDIT UNION,I am writing to file a complaint against Navy ...
6,1976,1976,3,1,Debt collection,"EQUIFAX, INC.",Could you please take a moment to review and e...
7,1473,1473,3,1,Debt collection,Experian Information Solutions Inc.,PLEASE BLOCKED AND REMOVED I respectfully requ...
8,1258,1258,1,3,Checking or savings account,"Block, Inc.",I am filing a complaint against Cash App ( Blo...
9,901,901,2,1,"Money transfer, virtual currency, or money ser...","Block, Inc.",I am filing a complaint against Cash App ( Blo...


In [36]:
print(
    "Repeated groups with multiple Complaint IDs:",
    (repeated_narrative_summary["unique_complaint_ids"] > 1).sum()
)

print(
    "Repeated groups spanning multiple companies:",
    (repeated_narrative_summary["unique_companies"] > 1).sum()
)

print(
    "Repeated groups spanning multiple products:",
    (repeated_narrative_summary["unique_products"] > 1).sum()
)

Repeated groups with multiple Complaint IDs: 5051
Repeated groups spanning multiple companies: 3051
Repeated groups spanning multiple products: 83


In [37]:
# Track repeated narratives without deleting complaints

narrative_frequency = narrative_comparison.map(
    narrative_comparison.value_counts()
)

df["narrative_repeat_count"] = (
    narrative_frequency
    .where(df["has_cleaned_narrative"])
    .astype("Int64")
)

df["is_repeated_narrative"] = (
    df["narrative_repeat_count"].gt(1).fillna(False)
)

print(
    "Rows flagged as repeated narratives:",
    df["is_repeated_narrative"].sum()
)

print(
    "Rows with unique narratives:",
    (
        df["has_cleaned_narrative"]
        & ~df["is_repeated_narrative"]
    ).sum()
)

Rows flagged as repeated narratives: 62453
Rows with unique narratives: 210013


In [38]:
record_validation = pd.Series({
    "total_rows": len(df),
    "unique_complaint_ids": df["complaint_id"].nunique(),
    "duplicate_complaint_ids": df["complaint_id"].duplicated().sum(),
    "rows_with_narratives": df["has_cleaned_narrative"].sum(),
    "repeated_narrative_rows": df["is_repeated_narrative"].sum()
})

display(record_validation.to_frame("count"))

,count
total_rows,566694
unique_complaint_ids,566694
duplicate_complaint_ids,0
rows_with_narratives,272466
repeated_narrative_rows,62453


### Repeated narrative decision

Although Complaint IDs were unique, 62,453 records belonged to 5,051
groups containing repeated narrative text. Every group contained multiple
Complaint IDs, while many repeated texts also appeared across different
companies or products.

Narrative equality alone was not considered sufficient evidence of
duplicate complaint records. These patterns may represent template-based
submissions, repeat filings, standardized complaint language, or
coordinated complaints. Consequently, the records were retained and
flagged using narrative repetition indicators. These flags can later be
used to prevent highly repeated templates from dominating text-focused
analysis.

In [39]:
# Inspect state values

print("Unique non-missing state values:", df["state"].nunique())

display(
    df["state"]
      .value_counts(dropna=False)
      .sort_index()
      .to_frame("complaint_count")
)

Unique non-missing state values: 61


,complaint_count
state,
AA,10
AE,167
AK,466
AL,9796
AP,109
...,...
WA,7051
WI,4417
WV,2307


In [40]:
# Inspect ZIP-code formatting

zip_values = df["zip_code"].astype("string").str.strip()

zip_validation = pd.Series({
    "missing_zip_codes": zip_values.isna().sum(),
    "five_digit_zip_codes":
        zip_values.str.fullmatch(r"\d{5}", na=False).sum(),
    "masked_zip_codes":
        zip_values.str.fullmatch(r"\d{3}XX", case=False, na=False).sum(),
    "other_formats":
        (
            zip_values.notna()
            & ~zip_values.str.fullmatch(
                r"(?:\d{5}|\d{3}XX)",
                case=False,
                na=False
            )
        ).sum()
})

display(zip_validation.to_frame("count"))

,count
missing_zip_codes,0
five_digit_zip_codes,488028
masked_zip_codes,71767
other_formats,6899


In [41]:
# Isolate ZIP codes that do not match expected formats

other_zip_mask = (
    zip_values.notna()
    & ~zip_values.str.fullmatch(
        r"(?:\d{5}|\d{3}XX)",
        case=False,
        na=False
    )
)

other_zip_values = zip_values[other_zip_mask]

print("Rows with other ZIP formats:", len(other_zip_values))
print("Unique other ZIP formats:", other_zip_values.nunique())

display(
    other_zip_values
      .value_counts(dropna=False)
      .head(30)
      .to_frame("count")
)

Rows with other ZIP formats: 6899
Unique other ZIP formats: 1


,count
zip_code,
XXXXX,6899


In [42]:
zip_format_diagnostics = pd.DataFrame({
    "zip_code": other_zip_values,
    "length": other_zip_values.str.len(),
    "contains_letters": other_zip_values.str.contains(
        r"[A-Za-z]", regex=True, na=False
    ),
    "contains_decimal": other_zip_values.str.contains(
        ".", regex=False, na=False
    ),
    "contains_hyphen": other_zip_values.str.contains(
        "-", regex=False, na=False
    )
})

display(
    zip_format_diagnostics["length"]
      .value_counts()
      .sort_index()
      .to_frame("row_count")
)

,row_count
length,
5,6899


In [43]:
display(
    zip_format_diagnostics
      .drop_duplicates("zip_code")
      .sort_values(["length", "zip_code"])
      .head(50)
)

,zip_code,length,contains_letters,contains_decimal,contains_hyphen
15,XXXXX,5,True,False,False


In [47]:
# Standardize ZIP codes while retaining CFPB privacy masking

df["zip_code"] = (
    df["zip_code"]
      .astype("string")
      .str.strip()
      .str.upper()
)

df["zip_code_status"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="string"
)

df.loc[
    df["zip_code"].str.fullmatch(r"\d{5}", na=False),
    "zip_code_status"
] = "complete"

df.loc[
    df["zip_code"].str.fullmatch(r"\d{3}XX", na=False),
    "zip_code_status"
] = "partially_masked"

df.loc[
    df["zip_code"].eq("XXXXX"),
    "zip_code_status"
] = "fully_masked"

In [48]:
# ZIP3 supports broader geographic analysis while respecting masking

df["zip3"] = df["zip_code"].str.extract(
    r"^(\d{3})",
    expand=False
).astype("string")

In [49]:
display(
    df["zip_code_status"]
      .value_counts(dropna=False)
      .to_frame("count")
)

print("Rows with available ZIP3:", df["zip3"].notna().sum())
print("Unclassified ZIP codes:", df["zip_code_status"].isna().sum())

,count
zip_code_status,
complete,488028
partially_masked,71767
fully_masked,6899


Rows with available ZIP3: 559795
Unclassified ZIP codes: 0


### ZIP-code handling

ZIP codes were retained as strings to preserve leading zeros and CFPB
privacy masking. The dataset contained 488,028 complete five-digit ZIP
codes, 71,767 partially masked codes, and 6,899 fully masked `XXXXX`
values. These masked values were treated as valid privacy-preserving
representations rather than data errors.

A separate ZIP3 field was derived where the first three digits were
available, supporting broader geographic analysis without imputing
unavailable location information.

In [50]:
for column in [
    "state",
    "submitted_via",
    "company_response_to_consumer",
    "timely_response"
]:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False))


--- state ---
state
TX    84779
FL    57926
CA    57746
GA    39340
NY    30432
      ...  
GU       23
AA       10
AS        4
MP        2
PW        1
Name: count, Length: 62, dtype: Int64

--- submitted_via ---
submitted_via
Web            546114
Referral         9220
Phone            8750
Postal mail      2610
Name: count, dtype: Int64

--- company_response_to_consumer ---
company_response_to_consumer
Closed with explanation            443349
Closed with non-monetary relief     97002
Closed with monetary relief         23196
Untimely response                    3120
In progress                            27
Name: count, dtype: Int64

--- timely_response ---
timely_response
Yes    556143
No      10551
Name: count, dtype: Int64


In [51]:
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# FINALIZE, SAMPLE, VALIDATE, AND SAVE
# --------------------------------------------------

RANDOM_STATE = 42
TARGET_PER_PRODUCT = 30_000

# Full cleaned dataset
cleaned_full_df = df.copy()

# Create balanced dataset:
# 30,000 rows from each product except Mortgage,
# for which all 24,697 rows are retained.
sampled_parts = []

for product, product_df in cleaned_full_df.groupby(
    "product",
    observed=True
):
    if product == "Mortgage":
        sampled_parts.append(product_df.copy())
    else:
        sampled_parts.append(
            product_df.sample(
                n=TARGET_PER_PRODUCT,
                random_state=RANDOM_STATE
            )
        )

balanced_df = (
    pd.concat(sampled_parts, ignore_index=True)
      .sample(frac=1, random_state=RANDOM_STATE)
      .reset_index(drop=True)
)

# Final validation
expected_counts = {
    "Debt collection": 30_000,
    "Credit card": 30_000,
    "Money transfer, virtual currency, or money service": 30_000,
    "Checking or savings account": 30_000,
    "Mortgage": 24_697
}

actual_counts = balanced_df["product"].value_counts().to_dict()

assert len(cleaned_full_df) == 566_694
assert cleaned_full_df["complaint_id"].nunique() == 566_694
assert len(balanced_df) == 144_697
assert balanced_df["complaint_id"].duplicated().sum() == 0
assert actual_counts == expected_counts

# Save outputs
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

full_output_path = output_dir / "cleaned_full_data.csv"
balanced_output_path = output_dir / "balanced_analytical_data.csv"

cleaned_full_df.to_csv(full_output_path, index=False)
balanced_df.to_csv(balanced_output_path, index=False)

print("NOTEBOOK 2 COMPLETE")
print(f"Full cleaned data: {cleaned_full_df.shape}")
print(f"Saved to: {full_output_path}")

print(f"\nBalanced analytical data: {balanced_df.shape}")
print(f"Saved to: {balanced_output_path}")

print("\nBalanced product counts:")
print(balanced_df["product"].value_counts())

NOTEBOOK 2 COMPLETE
Full cleaned data: (566694, 26)
Saved to: ../data/processed/cleaned_full_data.csv

Balanced analytical data: (144697, 26)
Saved to: ../data/processed/balanced_analytical_data.csv

Balanced product counts:
product
Money transfer, virtual currency, or money service    30000
Checking or savings account                           30000
Debt collection                                       30000
Credit card                                           30000
Mortgage                                              24697
Name: count, dtype: Int64


In [52]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save full cleaned dataset for SQL, trends and business analysis
df.to_csv(
    output_dir / "cleaned_full_data.csv",
    index=False
)

# Save balanced subset for balanced product comparisons
balanced_df.to_csv(
    output_dir / "balanced_analytical_data.csv",
    index=False
)

print("NOTEBOOK 2 COMPLETE")
print("Full cleaned dataset:", df.shape)
print("Balanced subset:", balanced_df.shape)

NOTEBOOK 2 COMPLETE
Full cleaned dataset: (566694, 26)
Balanced subset: (144697, 26)
